In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import os

# ✅ Fixed seed
torch.manual_seed(42)
np.random.seed(42)

mlflow.set_experiment("ecg-ssl-finetuning")


In [ ]:
BASELINE_RUN_NAME = "baseline-no-pretrain"

if mlflow.active_run() is not None:
    mlflow.end_run()

mlflow.start_run(run_name=BASELINE_RUN_NAME)
print(f"MLflow run started: {mlflow.active_run().info.run_id}")


In [ ]:
# ✅ Exact same data as Fine-tuning notebook — fair comparison
X_train = np.load("./ecg_ssl_research/data/processed/X_train.npy")
X_val   = np.load("./ecg_ssl_research/data/processed/X_val.npy")
X_test  = np.load("./ecg_ssl_research/data/processed/X_test.npy")
y_train = np.load("./ecg_ssl_research/data/processed/y_train.npy")
y_val   = np.load("./ecg_ssl_research/data/processed/y_val.npy")
y_test  = np.load("./ecg_ssl_research/data/processed/y_test.npy")

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val:   {X_val.shape},   {y_val.shape}")
print(f"Test:  {X_test.shape},  {y_test.shape}")

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights:", class_weights)

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = ECGDataset(X_train, y_train)
val_dataset   = ECGDataset(X_val,   y_val)
test_dataset  = ECGDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

In [ ]:
class ECGBaseline(nn.Module):
    """
    ✅ Identical architecture to ECGClassifier in Fine-tuning notebook
    ✅ Only difference — NO pretrained weights loaded
    ✅ This makes the comparison fair:
       Baseline    = same architecture, random init
       Fine-tuned  = same architecture, pretrained encoder
    """
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, 3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 3)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Random initialization — no pretrained weights
model = ECGBaseline().to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

print(f"Training on: {device}")
print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
epochs        = 40
best_val_loss = float('inf')
patience      = 10
counter       = 0

mlflow.log_params({
    "pretrained": False,
    "encoder_frozen": False,
    "classifier_layers": "128->64->32->3",
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": epochs,
    "optimizer": "Adam",
    "scheduler": "ReduceLROnPlateau",
    "scheduler_factor": 0.5,
    "scheduler_patience": 5,
    "patience": patience,
    "architecture": "Conv1D-Classifier",
    "dataset": "MIT-BIH",
    "class_weights": "balanced",
    "n_train_samples": len(X_train),
    "n_val_samples": len(X_val),
    "n_test_samples": len(X_test)
})

for epoch in range(epochs):

    # ========================
    # TRAINING PHASE
    # ========================
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)
        loss    = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ========================
    # VALIDATION PHASE
    # ========================
    model.eval()
    val_loss  = 0
    all_preds = []
    all_true  = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss    = criterion(outputs, y_batch)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(y_batch.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    # ========================
    # METRICS
    # ========================
    report_dict = classification_report(all_true, all_preds, output_dict=True)
    recall_N   = report_dict['0']['recall']
    recall_A   = report_dict['1']['recall']
    recall_V   = report_dict['2']['recall']
    accuracy   = report_dict['accuracy']
    macro_f1   = report_dict['macro avg']['f1-score']
    current_lr = optimizer.param_groups[0]['lr']

    mlflow.log_metrics({
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "accuracy": accuracy,
        "recall_N": recall_N,
        "recall_A": recall_A,
        "recall_V": recall_V,
        "macro_f1": macro_f1,
        "lr": current_lr
    }, step=epoch + 1)

    print(f"Epoch {epoch+1:>3} | "
          f"Train: {avg_train_loss:.4f} | "
          f"Val: {avg_val_loss:.4f} | "
          f"Acc: {accuracy:.4f} | "
          f"Recall A: {recall_A:.4f} | "
          f"Recall V: {recall_V:.4f} | "
          f"Macro F1: {macro_f1:.4f} | "
          f"LR: {current_lr:.2e}")

    scheduler.step(avg_val_loss)

    # ========================
    # EARLY STOPPING
    # ========================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        os.makedirs("./models", exist_ok=True)
        torch.save(model.state_dict(), "./models/best_baseline_model.pth")
        print("  ✅ Val loss improved → model saved")
    else:
        counter += 1
        print(f"  ⚠️  No improvement for {counter}/{patience} epochs")
        if counter >= patience:
            print("  🛑 Early stopping triggered!")
            break


In [ ]:
model.load_state_dict(
    torch.load("./models/best_baseline_model.pth",
               map_location=device, weights_only=True)
)
model.eval()

all_preds = []
all_true  = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y_batch.cpu().numpy())

cm = confusion_matrix(all_true, all_preds)
report_text = classification_report(
    all_true, all_preds,
    target_names=["Normal", "Afib (A)", "VTach (V)"]
)
report_dict = classification_report(
    all_true, all_preds,
    target_names=["Normal", "Afib (A)", "VTach (V)"],
    output_dict=True
)

print("=== Baseline: Confusion Matrix ===")
print(cm)
print()
print("=== Baseline: Classification Report ===")
print(report_text)

artifact_dir = "./mlflow_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=range(3),
    yticks=range(3),
    xticklabels=["Normal", "Afib (A)", "VTach (V)"],
    yticklabels=["Normal", "Afib (A)", "VTach (V)"],
    xlabel="Predicted label",
    ylabel="True label",
    title="Baseline Confusion Matrix"
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")

fig.tight_layout()
confusion_matrix_path = os.path.join(artifact_dir, "baseline_confusion_matrix.png")
fig.savefig(confusion_matrix_path, bbox_inches="tight")
mlflow.log_artifact(confusion_matrix_path)
plt.show()

classification_report_path = os.path.join(artifact_dir, "baseline_classification_report.txt")
with open(classification_report_path, "w") as f:
    f.write(report_text)
mlflow.log_artifact(classification_report_path)

mlflow.log_metrics({
    "test_accuracy": report_dict["accuracy"],
    "test_recall_N": report_dict["Normal"]["recall"],
    "test_recall_A": report_dict["Afib (A)"]["recall"],
    "test_recall_V": report_dict["VTach (V)"]["recall"],
    "test_macro_f1": report_dict["macro avg"]["f1-score"]
})

try:
    mlflow.pytorch.log_model(
        model,
        artifact_path="ecg_classifier",
        registered_model_name="ECG-Arrhythmia-Classifier"
    )
except Exception as exc:
    print(f"Model registry unavailable ({exc}). Logging classifier without registration.")
    mlflow.pytorch.log_model(model, artifact_path="ecg_classifier")

print(f"Run ID: {mlflow.active_run().info.run_id}")


In [ ]:
# ✅ Load fine-tuned model results for direct comparison
from torch.nn.functional import softmax

# reload fine-tuned model
class ECGClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 3)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

finetuned_model = ECGClassifier().to(device)
finetuned_model.load_state_dict(
    torch.load("./models/best_finetuned_model.pth",
               map_location=device, weights_only=True)
)
finetuned_model.eval()

ft_preds = []
ft_true  = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = finetuned_model(X_batch)
        _, preds = torch.max(outputs, 1)
        ft_preds.extend(preds.cpu().numpy())
        ft_true.extend(y_batch.numpy())

ft_report = classification_report(ft_true,  ft_preds, output_dict=True)
bl_report = classification_report(all_true, all_preds, output_dict=True)

comparison_lines = [
    "=" * 60,
    f"{'Metric':<25} {'Baseline':>15} {'SSL Finetuned':>15}",
    "=" * 60,
    f"{'Accuracy':<25} {bl_report['accuracy']:>15.4f} {ft_report['accuracy']:>15.4f}",
    f"{'Afib Precision':<25} {bl_report['1']['precision']:>15.4f} {ft_report['1']['precision']:>15.4f}",
    f"{'Afib Recall':<25} {bl_report['1']['recall']:>15.4f} {ft_report['1']['recall']:>15.4f}",
    f"{'Afib F1':<25} {bl_report['1']['f1-score']:>15.4f} {ft_report['1']['f1-score']:>15.4f}",
    f"{'VTach Precision':<25} {bl_report['2']['precision']:>15.4f} {ft_report['2']['precision']:>15.4f}",
    f"{'VTach Recall':<25} {bl_report['2']['recall']:>15.4f} {ft_report['2']['recall']:>15.4f}",
    f"{'VTach F1':<25} {bl_report['2']['f1-score']:>15.4f} {ft_report['2']['f1-score']:>15.4f}",
    f"{'Macro F1':<25} {bl_report['macro avg']['f1-score']:>15.4f} {ft_report['macro avg']['f1-score']:>15.4f}",
    "=" * 60,
]
comparison_text = "
".join(comparison_lines)
print(comparison_text)

artifact_dir = "./mlflow_artifacts"
os.makedirs(artifact_dir, exist_ok=True)
comparison_path = os.path.join(artifact_dir, "baseline_vs_ssl_comparison.txt")
with open(comparison_path, "w") as f:
    f.write(comparison_text)
mlflow.log_artifact(comparison_path)

mlflow.log_metrics({
    "ssl_accuracy_reference": ft_report['accuracy'],
    "accuracy_gain_over_baseline": ft_report['accuracy'] - bl_report['accuracy'],
    "afib_recall_gain_over_baseline": ft_report['1']['recall'] - bl_report['1']['recall'],
    "vtach_recall_gain_over_baseline": ft_report['2']['recall'] - bl_report['2']['recall'],
    "macro_f1_gain_over_baseline": ft_report['macro avg']['f1-score'] - bl_report['macro avg']['f1-score']
})


In [ ]:
if mlflow.active_run() is not None:
    mlflow.end_run()
    print("MLflow run closed.")
